In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
!ls /kaggle/input/q1-ka-ai-2026
!ls

In [ ]:
# Task 1: Write your code here:
df_food = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

print(f"Dataset shape: {df_food.shape}")
df_food.head()


In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()
df_food.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df_food.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df_food.describe()

In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)

# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data

df_food = df_food.drop("Order_ID",axis=1)
df_food

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df_food)

In [ ]:
# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean[cols].dropna()
print(f"After dropping missing price/year/odometer: {df_clean.shape}")

In [ ]:
df_clean.shape

In [ ]:
df_clean.info()

In [ ]:
# Task 3: Write your code here: Check and remove duplicates if any exist

# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
df_clean.shape

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)

# Encode features and target using LabelEncoder
categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print(categorical_cols)
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
# Scale features - fit on train, transform both

feature_cols = ['Courier_Experience_yrs', 'Courier_Experience_yrs', 'Courier_Experience_yrs', 'Courier_Experience_yrs' ]



# Standardize features using StandardScaler
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean


In [ ]:
df_clean

In [ ]:
# Task 6: Write your code here: #Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
#no need it's regression

In [ ]:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)
X

In [ ]:
X.info()

In [ ]:
y.info()

In [ ]:
y

In [ ]:
# Define classification models
models = {
    "Random Forest Classifier": RandomForestRegressor(n_estimators=100),
}
mae_scores = []


for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        mae_scores.append(mean_absolute_error(y_Test, y_pred))

    mae_scores = np.array(mae_scores)

    print(f"5-Fold CV Results:")
    print(f"MAE:  ${mae_scores.mean():,.2f}")

    # Print the results
    print("\n")


In [ ]:
y_pred

In [ ]:
# Task 1: Write your code here:

# Retrieve CatBoost feature importances and sort them
catboost_model = models["Random Forest Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:
# Year distribution
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black', color='orange')
plt.title('Year Distribution')
plt.xlabel('Year')
plt.ylabel('Frequency')
plt.show()

In [ ]:
!pip install catboost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score

In [ ]:
# Task Bonus: Write your code here:

# Define classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]), ('rf', models["Logistic Regression"]), ('gnb', models["Random Forest Classifier"])], voting="soft")
mae_scores = []
for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        mae_scores.append(mean_absolute_error(y_Test, y_pred))

    mae_scores = np.array(mae_scores)

    print(f"5-Fold CV Results:")
    print(f"MAE:  ${mae_scores.mean():,.2f}")

    # Print the results
    print("\n")

